In [2]:
!pip install transformers[sentencepiece] datasets scikit-learn tqdm matplotlib seaborn torch

import os
import re
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel 
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_name = "microsoft/deberta-v3-base"


if not os.path.exists('deberta_checkpoints/plots'):
    os.makedirs('deberta_checkpoints/plots')

print(f"Zariadenie: {device}")

Zariadenie: cuda


In [3]:
df = pd.read_csv('labeled_data.csv')

def clean_text(text):
    text = text.lower()
    text = re.sub(r'@[\w\-]+', '', text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'\brt\b', '', text)
    text = re.sub(r'&amp;|&#\d+;', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_tweet'] = df['tweet'].apply(clean_text)
df = df.drop_duplicates(subset=['clean_tweet'])
df = df[df['clean_tweet'] != ''].dropna()

X = df['clean_tweet']
y = df['class']

print(f"Záznamov pre DeBERTa: {len(df)}")

Záznamov pre DeBERTa: 24443


In [4]:
class SimpleDataset(Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.texts = texts.values
        self.labels = labels.values
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = self.tokenizer(
            text, 
            padding='max_length', 
            truncation=True, 
            max_length=64, 
            return_tensors='pt'
        )
        return {
            'ids': encoding['input_ids'].flatten(),
            'mask': encoding['attention_mask'].flatten(),
            'label': torch.tensor(self.labels[idx], dtype=torch.long)
        }

In [11]:
class DebertaClassifier(nn.Module):
    
    def __init__(self, num_classes=3):
        super(DebertaClassifier, self).__init__()
        self.deberta = AutoModel.from_pretrained(model_name)
        
        self.deberta = self.deberta.float() 
        
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(self.deberta.config.hidden_size, num_classes)
        
        self.classifier = self.classifier.float()

    def forward(self, ids, mask):
        outputs = self.deberta(input_ids=ids, attention_mask=mask)
        last_hidden_state = outputs.last_hidden_state
        
    
        cls_output = last_hidden_state[:, 0, :] 
        out = self.dropout(cls_output)
        
    
        return self.classifier(out)

In [12]:
def save_fold_loss_curve(history, fold_num):
    
    plt.figure(figsize=(4, 2.5))
    plt.plot(history['train'], label='Train Loss', linewidth=1.5, marker='o', markersize=4)
    plt.plot(history['val'], label='Val Loss', linewidth=1.5, marker='s', markersize=4)
    plt.title(f'DeBERTa Loss: Fold {fold_num}', fontsize=9)
    plt.xlabel('Epocha', fontsize=8)
    plt.ylabel('Loss', fontsize=8)
    plt.xticks(range(len(history['train'])), range(1, len(history['train']) + 1), fontsize=7)
    plt.legend(fontsize=7)
    plt.grid(True, linestyle='--', alpha=0.5)
    
    plot_path = f'deberta_checkpoints/plots/fold_{fold_num}_loss_curves.png'
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    plt.close()

In [13]:
def train_model_advanced(model, train_loader, val_loader, fold, epochs=5, patience=2):
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)
    criterion = torch.nn.CrossEntropyLoss()
    
    best_val_loss = float('inf')
    epochs_no_improve = 0  
    history = {'train': [], 'val': []}

    print(f"Trénovanie DeBERTa Fold {fold}")
    
    for epoch in range(epochs):
        model.train()
        total_train_loss = 0
        loop = tqdm(train_loader, desc=f"Fold {fold} | Ep {epoch+1}")
        
        for batch in loop:
            optimizer.zero_grad()
            ids, mask = batch['ids'].to(device), batch['mask'].to(device)
            labels = batch['label'].to(device)
            outputs = model(ids, mask)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_train_loss += loss.item()

        model.eval()
        total_val_loss, correct = 0, 0
        with torch.no_grad():
            for batch in val_loader:
                ids, mask = batch['ids'].to(device), batch['mask'].to(device)
                labels = batch['label'].to(device)
                outputs = model(ids, mask)
                total_val_loss += criterion(outputs, labels).item()
                correct += (torch.argmax(outputs, dim=1) == labels).sum().item()

        avg_train = total_train_loss / len(train_loader)
        avg_val = total_val_loss / len(val_loader)
        history['train'].append(avg_train)
        history['val'].append(avg_val)

        print(f"Ep {epoch+1}: Train {avg_train:.4f} | Val {avg_val:.4f} | Acc {correct/len(val_loader.dataset):.4f}")

        if avg_val < best_val_loss:
            best_val_loss = avg_val
            torch.save(model.state_dict(), f"deberta_checkpoints/best_model_fold_{fold}.pt")
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
        
        if epochs_no_improve >= patience:
            print("Early Stopping!")
            break
    return history

In [14]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
tokenizer = AutoTokenizer.from_pretrained(model_name)

global_best_loss = float('inf')
best_val_indices = None

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\n==================== DEBERTA FOLD {fold+1} ====================")
    
    train_ds = SimpleDataset(X.iloc[train_idx], y.iloc[train_idx], tokenizer)
    val_ds = SimpleDataset(X.iloc[val_idx], y.iloc[val_idx], tokenizer)
    
    train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=16)
    

    model = DebertaClassifier().to(device).float() 
    
    history = train_model_advanced(model, train_loader, val_loader, fold=fold+1)
    save_fold_loss_curve(history, fold_num=fold+1)
    
    path_to_current_best = f"deberta_checkpoints/best_model_fold_{fold+1}.pt"
    model.load_state_dict(torch.load(path_to_current_best))
    
    model.eval()
    current_fold_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            logits = model(batch['ids'].to(device), batch['mask'].to(device))
            loss = torch.nn.CrossEntropyLoss()(logits, batch['label'].to(device))
            current_fold_loss += loss.item()
    
    avg_current_fold_loss = current_fold_loss / len(val_loader)
    print(f"Finaálna strata pre Fold {fold+1}: {avg_current_fold_loss:.4f}")
    
    if avg_current_fold_loss < global_best_loss:
        global_best_loss = avg_current_fold_loss
        najlepsi_fold_cislo = fold + 1
        best_val_indices = val_idx
        torch.save(model.state_dict(), "deberta_checkpoints/absolute_best_deberta_model.pt")
        print(f"--> Nový absolútne najlepší model nájdený vo Folde {fold+1}!")


==================== DEBERTA FOLD 1 ====================


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Trénovanie DeBERTa Fold 1


Fold 1 | Ep 1:   0%|          | 0/1223 [00:00<?, ?it/s]

...

Ep 1: Train 0.3335 | Val 0.2486 | Acc 0.9147


Fold 5 | Ep 2:   0%|          | 0/1223 [00:00<?, ?it/s]

Ep 2: Train 0.2289 | Val 0.2576 | Acc 0.9073


Fold 5 | Ep 3:   0%|          | 0/1223 [00:00<?, ?it/s]

Ep 3: Train 0.2007 | Val 0.2526 | Acc 0.9114
Early Stopping!
Finaálna strata pre Fold 5: 0.2486


In [15]:
def get_detailed_report(model, loader, title="VÝSLEDOK"):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            logits = model(batch['ids'].to(device), batch['mask'].to(device))
            all_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
            all_labels.extend(batch['label'].numpy())
    
    print(f"\n=== {title} ===")
    print(classification_report(all_labels, all_preds, target_names=['Hate', 'Offensive', 'Neither'], digits=4))
    return all_labels, all_preds


final_model = DebertaClassifier().to(device)
final_model.load_state_dict(torch.load("deberta_checkpoints/absolute_best_deberta_model.pt"))
test_loader = DataLoader(SimpleDataset(X.iloc[best_val_indices], y.iloc[best_val_indices], tokenizer), batch_size=16)

y_true, y_pred = get_detailed_report(final_model, test_loader, "FINÁLNY REPORT: DeBERTa-v3")



Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



=== FINÁLNY REPORT: DeBERTa-v3 ===
              precision    recall  f1-score   support

        Hate     0.6923    0.3191    0.4369       282
   Offensive     0.9486    0.9696    0.9590      3786
     Neither     0.8763    0.9488    0.9111       821

    accuracy                         0.9286      4889
   macro avg     0.8391    0.7459    0.7690      4889
weighted avg     0.9217    0.9286    0.9208      4889

